# Silver Coreia — Espelho governado da Bronze

## Objetivo

Aplicar tipagem, padronização e governança aos dados
de chegadas de turistas internacionais à Coreia do Sul.

Fonte: Korea Tourism Data Lab
Período: 1990–2025

Regras:
- Preservar todos os registros da Bronze.
- Não aplicar filtros ou agregações de negócio.
- Não eliminar registros duplicados nesta etapa.
- Manter a rastreabilidade dos arquivos de origem.
- Validar os dados antes de materializar a Silver.

Tarefa especial: Unpivot da tabela.

In [0]:
import re
from pyspark.sql import functions as F

TABELA_BRONZE = "chegadas.coreia.bronze"
df_bronze = spark.table(TABELA_BRONZE)

Inspeção e qualidade de tipagem

In [0]:
print("Total de registros:", df_bronze.count())
print("Total de colunas:", len(df_bronze.columns))

In [0]:
display(
    spark.createDataFrame(
        [
            (i + 1, nome, tipo)
            for i, (nome, tipo)
            in enumerate(df_bronze.dtypes)
        ],
        ["posicao", "coluna", "tipo_dado"]
    )
)

Mapeamento das colunas e períodos

In [0]:
# Colunas que representam os períodos históricos
colunas_periodos = [
    c for c in df_bronze.columns
    if c.startswith("_c")
    and c[2:].isdigit()
    and int(c[2:]) >= 2
]

print("Colunas de períodos:", len(colunas_periodos))

# Identificar a linha de cabeçalho original
linha_cabecalho = (
    df_bronze
    .filter(F.col("_c0") == "Country")
    .select(*colunas_periodos)
    .take(2)
)

assert len(linha_cabecalho) == 1, (
    "Não foi possível identificar um cabeçalho único."
)

cabecalhos = linha_cabecalho[0].asDict()

print("Exemplos de cabeçalhos:")
for coluna in colunas_periodos[:5]:
    print(coluna, ":", cabecalhos[coluna])

In [0]:
mapa_periodos = []

for coluna in colunas_periodos:
    cabecalho = str(cabecalhos[coluna]).strip()

    # Reconhecer os formatos de ano e mês
    padrao = re.fullmatch(
        r"(\d{2,4})(?:년|\.)(?:(\d{1,2})월)?",
        cabecalho
    )

    if padrao is None:
        raise ValueError(
            f"Cabeçalho não reconhecido: {coluna} = {cabecalho}"
        )

    ano = int(padrao.group(1))

    # Interpretar os anos escritos com dois dígitos
    if ano < 100:
        ano = 2000 + ano if ano <= 30 else 1900 + ano

    mes = (
        int(padrao.group(2))
        if padrao.group(2)
        else None
    )

    granularidade = (
        "mensal" if mes is not None else "anual"
    )

    mapa_periodos.append((
        coluna,
        cabecalho,
        ano,
        mes,
        granularidade
    ))

# Criar DataFrame com a correspondência
df_periodos = spark.createDataFrame(
    mapa_periodos,
    schema="""
        coluna_origem STRING,
        cabecalho_original STRING,
        ano INT,
        mes INT,
        granularidade STRING
    """
)

display(df_periodos)

In [0]:
display(
    df_periodos
    .groupBy("granularidade")
    .count()
)

display(
    df_periodos
    .filter(F.col("ano") == 2025)
    .orderBy("mes")
)

Unpivot

In [0]:
df_longo = df_bronze.unpivot(
    ids=[
        "_c0",
        "_c1",
        "ingerido_em"
    ],
    values=colunas_periodos,
    variableColumnName="coluna_origem",
    valueColumnName="chegadas_brutas"
)

display(df_longo.limit(10))

In [0]:
df_silver_coreia = (
    df_longo
    .join(
        F.broadcast(df_periodos),
        on="coluna_origem",
        how="inner"
    )
    .select(
        F.col("_c0").alias("categoria_en"),
        F.col("_c1").alias("categoria_ko"),
        "ano",
        "mes",
        "granularidade",
        "chegadas_brutas",
        "coluna_origem",
        "cabecalho_original",
        "ingerido_em",
        F.current_timestamp().alias("_transformado_em")
    )
)

display(df_silver_coreia.limit(10))

In [0]:
linhas_bronze = df_bronze.count()
periodos = len(colunas_periodos)
linhas_silver = df_silver_coreia.count()

print("Linhas da Bronze:", linhas_bronze)
print("Colunas de períodos:", periodos)
print("Linhas esperadas:", linhas_bronze * periodos)
print("Linhas após o unpivot:", linhas_silver)

assert linhas_silver == linhas_bronze * periodos, (
    "A quantidade de registros não corresponde ao esperado."
)

Diagnóstico dos registros

In [0]:
display(
    df_bronze
    .select("_c0", "_c1")
    .distinct()
    .orderBy("_c0", "_c1")
)

In [0]:
display(
    df_silver_coreia.agg(
        F.count("*").alias("total_registros"),

        F.sum(
            F.when(
                F.col("chegadas_brutas").isNull(), 1
            ).otherwise(0)
        ).alias("chegadas_nulas"),

        F.sum(
            F.when(
                F.col("chegadas_brutas").isNotNull()
                & F.expr(
                    "try_cast(chegadas_brutas AS BIGINT)"
                ).isNull(),
                1
            ).otherwise(0)
        ).alias("chegadas_invalidas")
    )
)

In [0]:
display(
    df_silver_coreia
    .filter(
        F.col("chegadas_brutas").isNotNull()
        & F.expr(
            "try_cast(chegadas_brutas AS BIGINT)"
        ).isNull()
    )
    .groupBy("chegadas_brutas")
    .count()
    .orderBy(F.desc("count"))
)

Investigação numérica

In [0]:
df_diagnostico = (
    df_silver_coreia
    .withColumn(
        "valor",
        F.trim(F.col("chegadas_brutas"))
    )
    .withColumn(
        "formato",
        F.when(
            F.col("valor").isNull(),
            "NULL"
        )
        .when(
            F.col("valor") == "",
            "Vazio"
        )
        .when(
            F.col("valor") == "-",
            "Hífen"
        )
        .when(
            F.col("valor").rlike(r"^[0-9]+$"),
            "Inteiro simples"
        )
        .when(
            F.col("valor").rlike(
                r"^[0-9]{1,3}(,[0-9]{3})+$"
            ),
            "Separador de milhar"
        )
        .when(
            F.col("valor").rlike(
                r"^\([0-9,]+\)$"
            ),
            "Entre parênteses"
        )
        .otherwise("Outro formato")
    )
)

display(
    df_diagnostico
    .groupBy("formato")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
df_silver_coreia = (
    df_silver_coreia
    .withColumn(
        "chegadas",
        F.when(
            F.trim(F.col("chegadas_brutas")).rlike(
                r"^[0-9]+$|^[0-9]{1,3}(,[0-9]{3})+$"
            ),
            F.expr("""
                try_cast(
                    replace(chegadas_brutas, ',', '')
                    AS BIGINT
                )
            """)
        )
        .otherwise(F.lit(None).cast("bigint"))
    )
)

display(
    df_silver_coreia
    .filter(F.col("categoria_en") == "Brazil")
    .select(
        "categoria_en",
        "ano",
        "mes",
        "granularidade",
        "chegadas_brutas",
        "chegadas"
    )
    .orderBy("ano", "mes")
    .limit(20)
)

In [0]:
display(
    df_silver_coreia
    .filter(
        F.trim(F.col("chegadas_brutas")).rlike(
            r"^\([0-9,]+\)$"
        )
    )
    .select(
        "categoria_en",
        "categoria_ko",
        "ano",
        "mes",
        "granularidade",
        "chegadas_brutas"
    )
    .orderBy("ano", "categoria_en", "mes")
    .limit(30)
)

In [0]:
display(
    df_diagnostico
    .filter(F.col("formato") == "Outro formato")
    .groupBy("valor")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
)

In [0]:
display(
    df_diagnostico
    .filter(F.col("formato") == "Outro formato")
    .select(
        "categoria_en",
        "categoria_ko",
        "ano",
        "mes",
        "granularidade",
        "chegadas_brutas"
    )
    .orderBy("ano", "categoria_en")
    .limit(20)
)

In [0]:
display(
    df_diagnostico
    .filter(
        (F.col("formato") == "Outro formato") &
        (F.col("categoria_en") != "Country")
    )
    .select(
        "categoria_en",
        "categoria_ko",
        "ano",
        "mes",
        "granularidade",
        "chegadas_brutas"
    )
    .orderBy("ano", "categoria_en")
)

In [0]:
df_coreia_tratado = (
    df_silver_coreia
    .filter(
        F.col("categoria_en").isNull() |
        (F.col("categoria_en") != "Country")
    )
)

print(
    "Registros antes:",
    df_silver_coreia.count()
)

print(
    "Registros após retirar o cabeçalho:",
    df_coreia_tratado.count()
)

In [0]:
df_coreia_tratado = (
    df_coreia_tratado
    .withColumn(
        "formato_chegadas",
        F.when(
            F.col("chegadas_brutas").isNull(),
            "NULL"
        )
        .when(
            F.trim(F.col("chegadas_brutas")) == "",
            "Vazio"
        )
        .when(
            F.trim(F.col("chegadas_brutas")) == "-",
            "Hífen"
        )
        .when(
            F.trim(F.col("chegadas_brutas")).rlike(
                r"^[0-9]+$"
            ),
            "Inteiro simples"
        )
        .when(
            F.trim(F.col("chegadas_brutas")).rlike(
                r"^[0-9]{1,3}(,[0-9]{3})+$"
            ),
            "Separador de milhar"
        )
        .when(
            F.trim(F.col("chegadas_brutas")).rlike(
                r"^\([0-9,]+\)$"
            ),
            "Entre parênteses"
        )
        .when(
            F.trim(F.col("chegadas_brutas")).rlike(
                r"^-\([0-9,]+\)$"
            ),
            "Sinal negativo e parênteses"
        )
        .otherwise("Outro formato")
    )
)

In [0]:
display(
    df_coreia_tratado
    .groupBy("formato_chegadas")
    .count()
    .orderBy(F.desc("count"))
)

Classificação das categorias

In [0]:
df_coreia_tratado = (
    df_coreia_tratado
    .withColumn(
        "tipo_categoria",

        F.when(
            F.col("categoria_en").isNull()
            & F.col("categoria_ko").isNull(),
            "linha_vazia"
        )

        .when(
            F.col("categoria_en").startswith("*"),
            "nota"
        )

        .when(
            F.col("categoria_en").isin(
                "G.TOTAL",
                "Foreign Visitors"
            ),
            "total_geral"
        )

        .when(
            F.col("categoria_en").isin(
                "ASIA",
                "AFRICA",
                "Americas",
                "Europe",
                "Oceania",
                "Middle East Asia"
            ),
            "total_regional"
        )

        .when(
            F.col("categoria_en") == "others",
            "outros_regiao"
        )

        .when(
            F.col("categoria_ko") == "GCC6개국",
            "agrupamento_especial"
        )

        .when(
            F.col("categoria_en").isin(
                "Overseas Korean",
                "Stateless"
            ),
            "categoria_especial"
        )

        .otherwise(
            "pais_territorio_ou_categoria"
        )
    )
)

In [0]:
display(
    df_coreia_tratado
    .groupBy("tipo_categoria")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_coreia_tratado
    .filter(
        F.col("tipo_categoria") !=
        "pais_territorio_ou_categoria"
    )
    .select(
        "categoria_en",
        "categoria_ko",
        "tipo_categoria"
    )
    .distinct()
    .orderBy(
        "tipo_categoria",
        "categoria_en"
    )
)

In [0]:
print(
    "Registros após classificação:",
    df_coreia_tratado.count()
)

assert df_coreia_tratado.count() == 109242, (
    "A quantidade de registros foi alterada."
)

Validação da tipagem

In [0]:
display(
    df_coreia_tratado
    .groupBy("formato_chegadas")
    .agg(
        F.count("*").alias("total_registros"),

        F.count("chegadas").alias(
            "convertidos"
        ),

        F.sum(
            F.when(
                F.col("chegadas").isNull(), 1
            ).otherwise(0)
        ).alias("nao_convertidos")
    )
    .orderBy("formato_chegadas")
)

In [0]:
df_coreia_tratado.printSchema()

display(
    df_coreia_tratado
    .filter(F.col("categoria_en") == "Brazil")
    .filter(F.col("ano") == 2025)
    .select(
        "categoria_en",
        "categoria_ko",
        "ano",
        "mes",
        "granularidade",
        "chegadas_brutas",
        "chegadas",
        "tipo_categoria"
    )
    .orderBy(F.col("mes").asc_nulls_first())
)

Criação da tabela

In [0]:
TABELA_SILVER = "chegadas.silver.coreia"

(
    df_coreia_tratado.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_SILVER)
)

print("Tabela Silver Coreia criada!")

In [0]:
df_silver = spark.table(TABELA_SILVER)

total_silver = df_silver.count()

print("Total de registros:", total_silver)

assert total_silver == 109242, (
    "A quantidade de registros não corresponde ao esperado."
)

In [0]:
display(
    df_silver
    .groupBy("granularidade")
    .count()
    .orderBy("granularidade")
)

Documentação

In [0]:

COMENTARIOS_COREIA = {
    "categoria_en": "Nome do pais, territorio ou categoria estatistica conforme publicado na fonte, em ingles.",
    "categoria_ko": "Nome do pais, territorio ou categoria estatistica conforme publicado na fonte, em coreano.",
    "ano": "Ano de referencia do registro de entrada de visitantes na Coreia do Sul.",
    "mes": "Mes de referencia, de 1 a 12. Nulo nos registros de totais anuais.",
    "granularidade": "Identifica se o registro representa um total anual ou uma observacao mensal.",
    "chegadas_brutas": "Valor original da celula da planilha coreana, preservado como texto para auditoria.",
    "chegadas": "Quantidade de chegadas convertida para BIGINT quando o formato numerico foi identificado. Valores sem interpretacao confirmada permanecem nulos.",
    "coluna_origem": "Identificador da coluna original da Bronze antes do unpivot.",
    "cabecalho_original": "Cabecalho de ano ou mes associado a coluna original da planilha.",
    "ingerido_em": "Data e horario da ingestao original na camada Bronze.",
    "_transformado_em": "Data e horario de construcao do DataFrame tratado para a Silver.",
    "formato_chegadas": "Classificacao do formato original do valor de chegadas, utilizada para auditoria das conversoes.",
    "tipo_categoria": "Classificacao preliminar da linha da fonte: pais ou territorio, total regional, total geral, categoria especial, nota ou linha vazia."
}

for coluna, comentario in COMENTARIOS_COREIA.items():
    spark.sql(
        f"ALTER TABLE {TABELA_SILVER} "
        f"ALTER COLUMN `{coluna}` COMMENT '{comentario}'"
    )

print(f"{len(COMENTARIOS_COREIA)} colunas documentadas.")

In [0]:
spark.sql(f"""
COMMENT ON TABLE {TABELA_SILVER} IS
'Silver - Estatisticas de entradas internacionais
na Coreia do Sul. Dados reorganizados da estrutura
larga para longa por meio de unpivot.

Contem registros anuais e mensais, valores originais
de chegadas, conversoes numericas e classificacao
das categorias da fonte.

Os cabecalhos foram retirados da tabela de
observacoes e suas informacoes foram preservadas
nos metadados de periodo.

Valores entre parenteses e marcadores sem
interpretacao confirmada foram preservados
na coluna chegadas_brutas.

Fonte: Korea Tourism Organization.'
""")

In [0]:
display(
    df_silver.agg(
        F.count("*").alias("total_registros"),
        F.min("ano").alias("ano_inicial"),
        F.max("ano").alias("ano_final"),
        F.count("chegadas").alias("chegadas_convertidas"),
        F.sum(
            F.when(
                F.col("chegadas").isNull(), 1
            ).otherwise(0)
        ).alias("chegadas_nulas")
    )
)

display(
    df_silver
    .groupBy("tipo_categoria")
    .count()
    .orderBy(F.desc("count"))
)